[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/rainbow/rainbow.ipynb)

# The Rainbow

Sunlight entering a spherical raindrop is bent at its surface, reflected from the back of the drop and bent again on the way out, and comes back towards the sun turned by an angle that depends on how far from the drop's centre it entered. That angle has an extreme: light entering near one particular distance from the centre all leaves at nearly the same angle, and piles up there into a bright edge, the bow, about 42 degrees from the point straight opposite the sun. Light reflected twice inside has its own extreme near 51 degrees, the fainter secondary bow, and no light reflected once or twice leaves between the two: Alexander's dark band. Water bends blue light slightly more than red, so each colour has its bow at a slightly different angle.

Directions are modelled as antivectors, and so are the points of the drop's surface, as directions from its centre. Half a turn about the surface's normal, `normal >> direction`, reverses the part of a direction along the surface: `-(normal >> direction)` is the reflected direction, and the part along the surface is half the difference. Snell's law keeps that part times the refractive index, which holds both the law of sines and the light's staying in its plane of incidence. Inside a sphere every chord meets the surface at the same angle at both ends, so each point where the light meets the surface, and each chord, is the last one turned about the centre by the same rotor: the light's path inside the drop is the powers of one turn.

In the vector notation of geometric optics, Snell's law reads as $n_1 \sin\theta_1 = n_2 \sin\theta_2$ with the rays and the normal in one plane, and the deflection of light entering at incidence $i$ and refracted to $r$, after $k$ reflections inside, as $D = 2(i - r) + k(\pi - 2r)$, stationary where $\cos^2 i = \frac{n^2 - 1}{k(k + 2)}$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.optics.rainbow import render

np.set_printoptions(precision=2, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
# Directions, and the points of the drop's surface as directions from its centre, are antivectors,
# intrinsic; unit ones square to minus one.
Direction = ga.gatype.antivector()                                            # Antivector-
Rotor = ga.gatype.rotor()                                                     # Rotor+

# The direction sunlight travels, and the direction up, square to it.
forward, up = mv.x.dual(), mv.y.dual()                                        # [] Direction-, Direction-

## 1. Light through one drop

Light is followed through the drop at offsets from its centre across the drop's radius, square to the sunlight: in one colour for the drawing, and in every colour for the angles it leaves at. Where it leaves, the light is seen looking back against its direction, so the angle that matters is the one between that line of sight and the sunlight, whose direction points at the point straight opposite the sun.

The animation sweeps light across the drop, its path after one reflection orange and after two blue, with a bundle of light leaving behind it. Beside it, the angle each colour is seen at from the point opposite the sun, against the offset of the light: after one reflection it rises to its largest, the primary bow, and after two falls to its smallest, the secondary.

In [ ]:
# The wavelengths of the sunlight, in nanometres, and the refractive index of water for each: Cauchy's
# fit, a constant and a term falling with the wavelength squared, its coefficient in square nanometres. The
# index the swept light is drawn for, that of yellow light.
wavelengths = np.linspace(400, 700, 31)
index_constant = 1.3238
index_dispersion = 3140.0
drawn_index = 1.333
# Reflections inside the drop: once for the primary bow, twice for the secondary. Offsets of the light
# across the drop's radius, for the curves, the swept light and the bundle; the duration of a frame.
n_reflections = 2
n_rays = 4000
n_frames = 144
n_bundle = 48
duration_ms = 40

In [ ]:
def refract(direction: Direction, normal: Direction, ratio: np.ndarray) -> Direction:
    """The direction of light crossing a surface with unit `normal`, pointing into the medium it
    enters, which is `ratio` times as refractive as the one it leaves."""
    # Half a turn about the normal reverses the part of the direction along the surface; Snell's law
    # keeps that part times the refractive index.
    surface_part = (direction - (normal >> direction)) / (2 * ratio)          # [...] Direction-
    # Unit length fixes the part along the normal.
    return surface_part + normal * (1 - surface_part.scalar_norm_squared()).square_root()   # [...] Direction-


def entry(sunlight: Direction, offsets: Direction) -> Direction:
    """Where light along `sunlight`, passing the centre of the unit drop at each offset square to it,
    meets the drop: the point on its sunlit side, as a direction from the centre."""
    return offsets - sunlight * (1 - offsets.scalar_norm_squared()).square_root()   # [...] Direction-


def bounce(vertex: Direction, chord: Direction) -> Rotor:
    """The turn about the drop's centre carrying a point where light meets the surface from inside, and
    the chord arriving there, to the next point and the next chord."""
    # The next point is this one reflected across the plane through the centre square to the chord.
    next_vertex = -(chord >> vertex)                                          # [...] Direction-
    # The rotor taking one unit direction to another, in their plane.
    return (-(next_vertex * vertex)).square_root()                            # [...] Rotor+


def reflectances(offsets: Direction, index: np.ndarray) -> tuple[Scalar, Scalar]:
    """The fraction of light the surface reflects, polarized across and along the plane of incidence:
    Fresnel's coefficients, the same where the light enters as wherever it meets the surface again."""
    # The cosines of the angles of incidence and refraction; the offset is the sine of incidence.
    incidence = (1 - offsets.scalar_norm_squared()).square_root()             # [...] Scalar+
    refraction = (1 - offsets.scalar_norm_squared() / index**2).square_root() # [...] Scalar+
    reflectance_across = ((incidence - index * refraction) / (incidence + index * refraction)).squared()   # [...] Scalar+
    reflectance_along = ((refraction - index * incidence) / (refraction + index * incidence)).squared()   # [...] Scalar+
    return reflectance_across, reflectance_along


def paths(sunlight: Direction, offsets: Direction, index: np.ndarray, n_reflections: int) -> Iterator[tuple[Direction, Direction, Scalar]]:
    """Per number of reflections inside the drop, from none up to `n_reflections`: the point where the
    light leaves, the direction it leaves in, and the fraction of unpolarized sunlight leaving there."""
    vertex = entry(sunlight, offsets)                                         # [...] Direction-
    chord = refract(sunlight, -vertex, index)                                 # [...] Direction-
    # Every chord of a sphere meets it at the same angle at both ends: each point and chord inside is
    # the last one turned by the same rotor.
    turn = bounce(vertex, chord)                                              # [...] Rotor+
    reflectance_across, reflectance_along = reflectances(offsets, index)      # [...] Scalar+, Scalar+
    # The light transmitted into the drop and out of it again, in each polarization.
    passed_across, passed_along = (1 - reflectance_across).squared(), (1 - reflectance_along).squared()
    for _ in range(n_reflections + 1):
        vertex = turn >> vertex                                               # [...] Direction-
        yield vertex, refract(chord, vertex, 1 / index), (passed_across + passed_along) / 2
        chord = turn >> chord                                                 # [...] Direction-
        passed_across, passed_along = passed_across * reflectance_across, passed_along * reflectance_along


def sky(offsets: Direction) -> tuple[Scalar, Scalar]:
    """Per number of reflections, from one, and per colour, for light at each offset `[orders, colours,
    rays]`: the cosine of the angle from the point opposite the sun at which it is seen, and the sunlight
    it carries there, from the ring of offsets of its size."""
    indices = (index_constant + index_dispersion / wavelengths**2)[:, None]   # [colours, 1]
    _, *bows = paths(forward, offsets, indices, n_reflections)
    # Light leaving along `exit` is seen looking back against it; the point opposite the sun lies along
    # the sunlight.
    sight_cosines = stack([(-exit).dot(forward) for _, exit, _ in bows])      # [orders, colours, rays] Scalar+
    # The rings of offsets grow with their size.
    ring_sizes = offsets.norm()                                               # [rays] Scalar+
    return sight_cosines, stack([passed * ring_sizes for _, _, passed in bows])   # [orders, colours, rays] Scalar+


def sweep(offsets: Direction) -> tuple[Direction, Direction, Direction, Scalar]:
    """For light at each offset `[...]`: where it enters the drop; and after each number of
    reflections `[orders + 1, ...]`, the point where it leaves, the direction it leaves in, and the
    cosine of the angle from the point opposite the sun at which it is seen."""
    vertices, exits, _ = zip(*paths(forward, offsets, drawn_index, n_reflections))
    exits = stack(list(exits))                                                # [orders + 1, ...] Direction-
    return entry(forward, offsets), stack(list(vertices)), exits, (-exits).dot(forward)


# Offsets at the middle of equal steps across the drop's radius.
offsets = up * ((np.arange(n_rays) + 0.5) / n_rays)                           # [rays] Direction-
sight_cosines, flux = sky(offsets)                                            # [orders, colours, rays] Scalar+, Scalar+
swept_light = sweep(up * ((np.arange(n_frames) + 0.5) / n_frames))
bundle = sweep(up * ((np.arange(n_bundle) + 0.5) / n_bundle))
print("primary bow, violet to red, degrees:  ", np.degrees(np.arccos(sight_cosines.to_array()[0].min(axis=-1)))[[0, -1]])
print("secondary bow, violet to red, degrees:", np.degrees(np.arccos(sight_cosines.to_array()[1].max(axis=-1)))[[0, -1]])
show_animation(render.animate_sweep(swept_light, bundle, offsets, sight_cosines, wavelengths), "rainbow_sweep", duration_ms)

## 2. The bows on the sky

Every drop sends light back the same way, so the light of each colour arrives from a ring around the point opposite the sun: the bow is a circle about it, and the horizon cuts it off by the sun's height. Light entering at each offset belongs to a ring of offsets as large as the offset, so it carries that much sunlight, times the fraction Fresnel's coefficients pass; spread over the sky, the light per solid angle is that sunlight over the size of the ring of sky it arrives in. The primary bow is red outside and violet inside, the secondary reversed and fainter, with light filling the sky inside the primary and outside the secondary and the dark band between them. The sun's disc, half a degree wide, blurs each colour's edge by its width.

In [ ]:
# The sun's height above the horizon, and the width of its disc, in degrees.
sun_elevation = 12.0
sun_width = 0.53

In [ ]:
save_figure(render.draw_sky(sight_cosines, flux, wavelengths, sun_elevation, sun_width), "rainbow_sky");

In [ ]:
# checks
# Each bow lies where the angle of the light leaving is stationary, Descartes's angle: after one
# reflection the largest from the point opposite the sun, after two the smallest.
index = index_constant + index_dispersion / wavelengths**2
reflections = np.arange(1, n_reflections + 1)[:, None]                        # [orders, 1]
incidence = np.arccos(np.sqrt((index**2 - 1) / (reflections * (reflections + 2))))
refraction = np.arcsin(np.sin(incidence) / index)
deflection = 2 * (incidence - refraction) + reflections * (np.pi - 2 * refraction)
cosines = sight_cosines.to_array()                                            # [orders, colours, rays]
np.testing.assert_allclose([cosines[0].min(axis=-1), cosines[1].max(axis=-1)], -np.cos(deflection), atol=1e-6)
# The light leaving is a unit direction, and each fraction of sunlight is less after more reflections.
_, _, exits, _ = swept_light
np.testing.assert_allclose(exits.scalar_norm_squared().to_array(), 1, atol=1e-6)
np.testing.assert_array_less(flux.to_array()[1], flux.to_array()[0])